In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

In [2]:
df = pd.read_csv("../data/sncf_retards.csv", sep=";")

# Supprimer les colonnes inutiles (commentaires)
cols_to_drop = [c for c in df.columns if "Commentaire" in c]
df = df.drop(columns=cols_to_drop)

# Supprimer la colonne Service (toujours "National")
df = df.drop(columns=["Service"])

# Extraire année et mois
df["Année"] = df["Date"].str[:4].astype(int)
df["Mois"] = df["Date"].str[5:7].astype(int)
df = df.drop(columns=["Date"])

# Variable cible
target = "Retard moyen de tous les trains à l'arrivée"

# Filtrer les valeurs aberrantes (retard négatif = erreur)
print(f"Avant nettoyage: {len(df)} lignes")
df = df[df[target] >= 0]
print(f"Après nettoyage: {len(df)} lignes")

df.head()

Avant nettoyage: 12181 lignes
Après nettoyage: 12061 lignes


,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,Nombre de trains en retard à l'arrivée,Retard moyen des trains en retard à l'arrivée,...,Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)",Année,Mois
0,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141,870,5,289,11.247809,3.693179,147,28.436735,...,44,8,36.134454,31.092437,10.924370,15.966387,5.042017,0.840336,2018,1
1,LE MANS,PARIS MONTPARNASSE,56,406,1,213,8.479969,4.567119,105,18.049048,...,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333,2018,1
2,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166,226,0,21,6.239683,0.286283,19,24.736842,...,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111,2018,1
3,PARIS MONTPARNASSE,NANTES,124,508,3,71,7.235211,0.734290,58,33.726437,...,18,8,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852,2018,1
4,POITIERS,PARIS MONTPARNASSE,94,472,4,224,6.784673,3.229701,89,14.592697,...,10,0,15.789474,45.614035,19.298246,15.789474,1.754386,1.754386,2018,1


In [3]:
col_arrivee = "Gare d'arrivée"

le_depart = LabelEncoder()
le_arrivee = LabelEncoder()

df["Gare_depart_enc"] = le_depart.fit_transform(df["Gare de départ"])
df["Gare_arrivee_enc"] = le_arrivee.fit_transform(df[col_arrivee])

# Sauvegarder les encoders pour Streamlit plus tard
joblib.dump(le_depart, "../models/le_depart.joblib")
joblib.dump(le_arrivee, "../models/le_arrivee.joblib")

print(f"Gares de départ uniques: {len(le_depart.classes_)}")
print(f"Gares d'arrivée uniques: {len(le_arrivee.classes_)}")

Gares de départ uniques: 59
Gares d'arrivée uniques: 59


In [4]:
features = [
    "Gare_depart_enc",
    "Gare_arrivee_enc",
    "Durée moyenne du trajet",
    "Nombre de circulations prévues",
    "Nombre de trains annulés",
    "Année",
    "Mois",
]

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]} lignes | Test: {X_test.shape[0]} lignes")

Train: 9648 lignes | Test: 2413 lignes


In [5]:
models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "gradient_boosting": GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42),
}

for name, model in models.items():
    print(f"Entraînement de {name}...")
    model.fit(X_train, y_train)
    
    # Sauvegarder le modèle
    path = f"../models/{name}.joblib"
    joblib.dump(model, path)
    print(f"  → Sauvegardé dans {path}")

print("\nTous les modèles sont entraînés et sauvegardés.")

Entraînement de linear_regression...
  → Sauvegardé dans ../models/linear_regression.joblib
Entraînement de random_forest...
  → Sauvegardé dans ../models/random_forest.joblib
Entraînement de gradient_boosting...
  → Sauvegardé dans ../models/gradient_boosting.joblib

Tous les modèles sont entraînés et sauvegardés.


In [6]:
print(f"{'Modèle':<25} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print("-" * 55)

for name, model in models.items():
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"{name:<25} {mae:>8.2f} {rmse:>8.2f} {r2:>8.4f}")

Modèle                         MAE     RMSE       R²
-------------------------------------------------------
linear_regression             2.66     3.59   0.1482
random_forest                 1.89     2.77   0.4928
gradient_boosting             1.89     2.74   0.5038


In [7]:
!pip install xgboost

In [9]:
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import AdaBoostRegressor

models_v2 = {
    "adaboost": AdaBoostRegressor(n_estimators=200, random_state=42),
    "svr": Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel="rbf", C=10))]),
    "knn": Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor(n_neighbors=10))]),
}

for name, model in models_v2.items():
    print(f"Entraînement de {name}...")
    model.fit(X_train, y_train)
    path = f"../models/{name}.joblib"
    joblib.dump(model, path)
    print(f"  → Sauvegardé dans {path}")

Entraînement de adaboost...
  → Sauvegardé dans ../models/adaboost.joblib
Entraînement de svr...
  → Sauvegardé dans ../models/svr.joblib
Entraînement de knn...
  → Sauvegardé dans ../models/knn.joblib


In [10]:
all_models = {**models, **models_v2}

print(f"{'Modèle':<25} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print("-" * 55)

results = []
for name, model in all_models.items():
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"{name:<25} {mae:>8.2f} {rmse:>8.2f} {r2:>8.4f}")
    results.append({"Modèle": name, "MAE": mae, "RMSE": rmse, "R²": r2})

results_df = pd.DataFrame(results).sort_values("R²", ascending=False)
print("\nClassement par R² :")
print(results_df.to_string(index=False))

Modèle                         MAE     RMSE       R²
-------------------------------------------------------
linear_regression             2.66     3.59   0.1482
random_forest                 1.89     2.77   0.4928
gradient_boosting             1.89     2.74   0.5038
adaboost                      3.12     3.96  -0.0330
svr                           2.18     3.17   0.3389
knn                           2.17     3.06   0.3805

Classement par R² :
           Modèle      MAE     RMSE        R²
gradient_boosting 1.890641 2.742439  0.503760
    random_forest 1.891746 2.772472  0.492831
              knn 2.166465 3.064224  0.380475
              svr 2.185000 3.165381  0.338896
linear_regression 2.658654 3.592985  0.148218
         adaboost 3.119930 3.956817 -0.033023
